# Tahap 2 — Canonical Twin State Demonstration

Notebook ini mendemonstrasikan transformasi satu raw telemetry record menjadi Canonical Twin State. Evaluasi empiris seluruh dataset tetap menjadi ruang lingkup Notebook 05; notebook ini tidak mengulang evaluasi 2.027.520 record.

In [ ]:
# 2. Environment check
from pathlib import Path
import importlib.metadata
import os
import platform
import sys

EXPECTED_DATASET_SHA256 = 'ca7831a188a191edbf82a673fac90dbb875b5095986ed07699c02530f2a02a0e'
REPOSITORY_URL = 'https://github.com/rehanalfarizu/new_jurnal.git'
IS_COLAB = 'google.colab' in sys.modules

def find_repository_root():
    candidates = [Path.cwd(), Path.cwd() / 'new_jurnal', Path.cwd().parent]
    for candidate in candidates:
        if (candidate / 'configs' / 'experiment.yaml').is_file() and (candidate / 'src').is_dir():
            return candidate.resolve()
    return None

REPO_ROOT = find_repository_root()
print({'python': sys.version.split()[0], 'platform': platform.platform(), 'colab': IS_COLAB, 'repository_found': REPO_ROOT is not None})

In [ ]:
# 3. Google Colab setup
import subprocess

if IS_COLAB and REPO_ROOT is None:
    clone_target = Path('/content/new_jurnal')
    if not clone_target.exists():
        subprocess.run(['git', 'clone', REPOSITORY_URL, str(clone_target)], check=True)
    REPO_ROOT = clone_target.resolve()
elif REPO_ROOT is None:
    raise RuntimeError('Repository tidak ditemukan. Jalankan notebook dari root new_jurnal atau direktori notebooks/.')
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print('Repository aktif:', REPO_ROOT.name)

In [ ]:
# 4. Dependency setup
import importlib.util

if IS_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
required_modules = {'pandas': 'pandas', 'PyYAML': 'yaml'}
missing = [name for name, module in required_modules.items() if importlib.util.find_spec(module) is None]
if missing:
    raise ModuleNotFoundError('Dependency kernel belum lengkap: ' + ', '.join(missing))
print('Dependency requirements.txt tersedia pada kernel aktif.')

In [ ]:
# 5. Dataset path + checksum
import hashlib
# Google Colab opsional:
# from google.colab import drive
# drive.mount('/content/drive')
# os.environ['SENSOR_DATA_PATH'] = '/content/drive/MyDrive/.../sensor_data.csv'
configured_path = os.environ.get('SENSOR_DATA_PATH', '').strip()
default_path = (REPO_ROOT / 'data/raw/sensor_data.csv').resolve()
if configured_path:
    DATASET_PATH = Path(configured_path).expanduser()
    if not DATASET_PATH.is_absolute():
        DATASET_PATH = (REPO_ROOT / DATASET_PATH).resolve()
    path_source = 'SENSOR_DATA_PATH'
elif default_path.is_file():
    DATASET_PATH, path_source = default_path, 'data/raw/sensor_data.csv'
elif not IS_COLAB:
    candidates_by_file = {}
    for pattern in ('*/Data/sensor_data.csv', '*/data/sensor_data.csv'):
        for path in REPO_ROOT.parent.glob(pattern):
            if path.is_file():
                info = path.stat()
                candidates_by_file[(info.st_dev, info.st_ino)] = path.resolve()
    candidates = sorted(candidates_by_file.values())
    if len(candidates) > 1:
        raise RuntimeError('Lebih dari satu dataset ditemukan; tetapkan SENSOR_DATA_PATH secara eksplisit.')
    DATASET_PATH = candidates[0] if candidates else default_path
    path_source = 'kandidat tunggal folder proyek saudara' if candidates else 'data/raw/sensor_data.csv'
else:
    DATASET_PATH, path_source = default_path, 'data/raw/sensor_data.csv'
if not DATASET_PATH.is_file():
    raise FileNotFoundError('sensor_data.csv tidak ditemukan. Tetapkan SENSOR_DATA_PATH atau mount Google Drive.')
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()
DATASET_SHA256 = sha256_file(DATASET_PATH)
if DATASET_SHA256 != EXPECTED_DATASET_SHA256:
    raise ValueError(f'Checksum dataset tidak sesuai: {DATASET_SHA256}')
print('Sumber resolusi dataset:', path_source)
print('SHA-256 terverifikasi:', DATASET_SHA256)

In [ ]:
# 6. Load canonical configuration
import pandas as pd
import yaml
from IPython.display import display

with Path('configs/experiment.yaml').open(encoding='utf-8') as handle:
    experiment_config = yaml.safe_load(handle)
canonical_config = experiment_config['canonical_state']
display(pd.DataFrame([
    {'setting': 'schema_version', 'value': canonical_config['schema_version']},
    {'setting': 'room_id', 'value': canonical_config['room_id']},
    {'setting': 'staleness_seconds', 'value': canonical_config['staleness_seconds']},
    {'setting': 'source_timezone', 'value': experiment_config['data']['source_timezone']},
    {'setting': 'timezone_policy', 'value': experiment_config['preprocessing']['timezone_policy']},
]))

In [ ]:
# 7. Representative raw record
import csv
with DATASET_PATH.open(encoding='utf-8-sig', newline='') as handle:
    raw_record = next(csv.DictReader(handle))
display(pd.DataFrame([raw_record]))
print('Record pertama dipakai sebagai demonstrasi; tidak ada sampling outcome atau evaluasi seluruh dataset.')

In [ ]:
# 8. Raw-to-canonical transformation
from src.twin_state.canonical import CanonicalStateTransformer, parse_timestamp_utc, validate_canonical_state_schema
transformer = CanonicalStateTransformer(
    room_id=canonical_config['room_id'],
    schema_version=canonical_config['schema_version'],
    validation_ranges=canonical_config['validation_ranges'],
)
canonical_state = transformer.transform(
    raw_record, source_row_number=2, source_file_sha256=DATASET_SHA256
)
print('Transformer source of truth:', transformer.__class__.__module__)

In [ ]:
# 9. Canonical state structure
import json
from IPython.display import JSON
display(JSON(canonical_state, expanded=True))

In [ ]:
# 10. Source-to-state mapping
field_mapping = pd.read_csv('results/tables/canonical_field_mapping.csv')
display(field_mapping[['source_field', 'canonical_field', 'transformation', 'value_preservation', 'mapping_status']])

In [ ]:
# 11. Field type dan unit
display(field_mapping[['source_field', 'canonical_field', 'source_type', 'canonical_type', 'unit', 'validation_rule']])

In [ ]:
# 12. Data-quality flags
display(pd.DataFrame([{
    'valid': canonical_state['data_quality']['valid'],
    'flags': canonical_state['data_quality']['flags'],
}]))
print('room_id_unresolved adalah limitation metadata dan tidak menghapus telemetry.')

In [ ]:
# 13. UTC normalization
raw_timestamp = raw_record['Timestamp']
parsed_timestamp, timestamp_policy = parse_timestamp_utc(raw_timestamp)
display(pd.DataFrame([{
    'source_timestamp_text': raw_timestamp,
    'timestamp_utc': canonical_state['timestamp_utc'],
    'timestamp_policy': timestamp_policy,
    'clock_value_shifted': False,
}]))

In [ ]:
# 14. Provenance
display(pd.DataFrame([canonical_state['provenance']]))
print('Checksum dan source row menghubungkan state demonstrasi kembali ke record sumber.')

In [ ]:
# 15. room_id = unresolved
assert canonical_state['room_id'] == 'unresolved'
print('room_id:', canonical_state['room_id'])
print('Identitas ruang tidak dibuat karena tidak tersedia pada raw dataset.')

In [ ]:
# 16. staleness_seconds = None
assert canonical_state['data_quality']['staleness_seconds'] is None
print('staleness_seconds:', canonical_state['data_quality']['staleness_seconds'])
print('Tidak tersedia timestamp kamera independen untuk menghitung staleness.')

In [ ]:
# 17. Validation example
schema_violations = validate_canonical_state_schema(canonical_state)
display(pd.DataFrame([{
    'schema_conforming': len(schema_violations) == 0,
    'violation_count': len(schema_violations),
    'violations': schema_violations,
    'data_quality_valid': canonical_state['data_quality']['valid'],
}]))
if schema_violations:
    raise AssertionError(schema_violations)

In [ ]:
# 18. Reproducibility summary
summary = {
    'dataset_sha256': DATASET_SHA256,
    'source_row_number': canonical_state['provenance']['source_row_number'],
    'schema_version': canonical_state['schema_version'],
    'transformer': CanonicalStateTransformer.__module__ + '.' + CanonicalStateTransformer.__name__,
    'schema_validator': validate_canonical_state_schema.__module__ + '.' + validate_canonical_state_schema.__name__,
    'full_dataset_evaluation_repeated': False,
    'empirical_evaluation_notebook': 'notebooks/05_digital_twin_evaluation.ipynb',
}
display(summary)